# Лабораторная работа № 5 — Разведочный анализ и проверка качества данных

**ДГТУ · МИК21**  
Дисциплина: «Проект — применение машинного обучения в креативных индустриях»  
**Проект:** CineMatch — рекомендации фильмов по истории оценок и жанрам.  
**Набор:** MovieLens Latest Small (GroupLens), продолжение Data Card v1 из ЛР № 4.

**Цель:** выявить ошибки, смещения и утечки, которые могут испортить обучение или оценку рекомендательной системы.

**Повторный запуск:** открыть [ноутбук в Colab](https://colab.research.google.com/github/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/Lab5_EDA_CineMatch_MIK21.ipynb) → «Выполнить все». При наличии сети он скачает официальные CSV и пересчитает таблицы, примеры и графики. Архив MovieLens в репозиторий не помещён.

> Ниже приведены восемь графиков из зафиксированного **агрегированного среза** с Git blob SHA для MovieLens. Они доступны при просмотре на GitHub до выполнения ноутбука. В конце расположены ячейки, которые заново строят графики и checks по полным строкам CSV. Агрегированный срез не подменяет повторный расчёт исходных данных.

## 1. Вопросы EDA и решения

| № | Что проверяем | Зачем / возможное решение |
|---|---|---|
| 1 | Какие жанры доминируют? | Не трактовать жанровые метрики без поддержки редких жанров |
| 2 | Как распределены оценки 0.5–5? | Выбирать baseline и учитывать смещение в сторону высоких оценок |
| 3 | Какие десятилетия представлены? | Ограничить переносимость на современные релизы |
| 4 | Когда выставлялись оценки? | Проверить временную актуальность и выбрать хронологический split |
| 5 | Сколько жанров у фильма? | Корректно обрабатывать многометочные объекты и «без жанров» |
| 6 | Есть ли дефектные метаданные и дубли? | Ввести quality gates и журнал исключений |
| 7 | Не разрывает ли split связанные по времени взаимодействия? | Группировать одинаковые timestamps внутри пользователя |
| 8 | Насколько разрежена история фильмов? | Оценивать холодный старт и длинный хвост отдельно |

**Единицы счёта:** жанры — количество фильмов с данным тегом (один фильм может учитываться несколько раз); оценки — взаимодействия; годы — год оценки; десятилетия — год, извлечённый из названия фильма.

### График 1. Жанры

![График 1](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/01_genres.svg)

**Интерпретация и действие:** Жанры: разница между частотами Drama и Film-Noir показывает смещение каталога. Решение — срезы качества по редким жанрам, не полагаться только на глобальную метрику.

### График 2. Оценки

![График 2](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/02_ratings.svg)

**Интерпретация и действие:** Оценки: высокие и средние баллы преобладают; отсутствие оценки нельзя объявлять отрицательным предпочтением. Решение — явно определить порог позитивной релевантности (например, ≥ 4).

### График 3. Десятилетия

![График 3](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/03_release_decades.svg)

**Интерпретация и действие:** Релизы: крупная доля каталога относится к 1990–2010-м. Решение — не распространять выводы на новые релизы без дополнительной проверки.

### График 4. Годы оценок

![График 4](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/04_rating_years.svg)

**Интерпретация и действие:** История: оценки относятся к 1996–2018 годам, активность по годам неравномерна. Решение — использовать время взаимодействия и отдельно обсуждать временную переносимость.

### График 5. Число жанров

![График 5](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/05_genres_per_movie.svg)

**Интерпретация и действие:** Многожанровость: для фильмов обычны 2–3 жанра, 34 фильма вовсе без меток. Решение — многометочное кодирование, неизвестные жанры учитывать отдельно.

### График 6. Аномалии

![График 6](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/06_quality_issues.svg)

**Интерпретация и действие:** Качество: найдены отсутствующие жанры, год и TMDb ID, отдельные коллизии заголовков. Решение — явно различать критические и некритические поля; не удалять фильмы без жанров молча.

### График 7. Разбиение

![График 7](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/07_split.svg)

**Интерпретация и действие:** Разделение: приблизительно 62 тыс. train, 19 тыс. validation и 20 тыс. test взаимодействий. Равенство или доли сами по себе не доказывают отсутствие утечки — проверить группы.

### График 8. Популярность

![График 8](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/figures/08_popularity.svg)

**Интерпретация и действие:** Длинный хвост: 3 446 фильмов получили всего одну оценку. Решение — cold-start fallback и охват каталога анализировать отдельно.

## 2. Паспорт агрегированного среза и проверка качества

Зафиксированные идентификаторы публичного зеркала `smanihwr/ml-latest-small`: movies `2c1b2e74fe24d0fc851aa9745702fcd7777f690a`, ratings `51974adb19938bda0e00b2d8c051438fb33893f0`, tags `224e0c167e6cdd71ec4c6fd9da208ee5a286caaf`, links `d5ccfdf196ebe5b7bbd09d505ea8ca263ecc33ae`. Итоги среза приведены ниже; отдельные вычисления полного архива выполняются в ячейках далее.

| Проверка | Результат агрегированного среза | Решение |
|---|---:|---|
| Фильмы / оценки / теги / внешние ссылки | 9 742 / 100 836 / 3 683 / 9 742 | Сверить объёмы и идентификаторы в live-аудите |
| Пользователи | 610 | Оценивать существующих пользователей, не заявлять полное покрытие аудитории |
| Полностью повторные оценки | 0 | Сохранить автоматический контроль |
| Повторные пары `userId,movieId` | 0 | Сохранить контроль до split |
| Оценки с неизвестным movieId | 0 | Проверить внешние ключи |
| Фильмы без жанров | 34 | Отдельный сегмент; не выдавать фиктивный жанр |
| Заголовки без года | 13 | Не делать год обязательным признаком |
| Отсутствующий `tmdbId` | 8 | Только необязательная ссылка |
| Коллизии названий | 5 | Использовать `movieId` как ключ |
| Рейтинг вне допустимого диапазона | 0 | Quality gate |
| Оценки раньше года релиза из названия | 3 | Отдельно разобрать ошибки года/времени; не удалять автоматически |
| Группы одинаковых `userId,timestamp` с ≥2 оценками | 7 438 групп / 23 228 оценок | Не дробить группы между выборками |
| Отсутствующие `tags.tag` | 0 | Контроль пользовательского текста и прав |

**Важно:** отсутствующий `tmdbId` не влияет на baseline по `genres`; неизвестный жанр препятствует content-based сигналу, но фильм может участвовать в популярностных рекомендациях.

## 3. Репрезентативные и проблемные карточки

| Категория | movieId | title | genres | Решение |
|---|---:|---|---|---|
| Обычная | 1 | Toy Story (1995) | Adventure / Animation / Children / Comedy / Fantasy | многометочное представление |
| Обычная | 2 | Jumanji (1995) | Adventure / Children / Fantasy | обычная карточка |
| Без жанров | 114335 | La cravate (1957) | (no genres listed) | сегмент качества; не придумывать жанр |
| Без жанров | 122888 | Ben-hur (2016) | (no genres listed) | отдельно проверить поиск |
| Без года | 40697 | Babylon 5 | Sci-Fi | оставить movieId, год nullable |
| Без года | 140956 | Ready Player One | Action / Sci-Fi / Thriller | не делать парсинг года обязательным |
| Коллизия названий | 838 и 26958 | Emma (1996) | разные жанры | не объединять по title |

Это примеры **текстовых карточек** MovieLens. Афиши или кадры в исходном наборе отсутствуют, поэтому просмотр медиа-изображений невозможен без отдельного разрешённого источника. Вместо них осматриваются как типичные, так и неудобные карточки.

## 4. Полностью воспроизводимый построчный аудит

Следующие ячейки скачивают и **проверяют** CSV, заново рассчитывают показатели, сохраняют таблицу качества и строят 8 графиков из реальных строк.

Для запуска нужна сеть (официальный GroupLens либо публичное зеркало с проверкой SHA). Если сеть отсутствует, результаты выше остаются доступными как опубликованный агрегированный срез, но **не считаются результатом повторного запуска**.

In [ ]:
import hashlib, io, zipfile, re
from pathlib import Path
from urllib.request import urlopen, Request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NAMES = ("movies.csv", "ratings.csv", "tags.csv", "links.csv")
GIT_SHA = {
    "movies.csv": "2c1b2e74fe24d0fc851aa9745702fcd7777f690a",
    "ratings.csv": "51974adb19938bda0e00b2d8c051438fb33893f0",
    "tags.csv": "224e0c167e6cdd71ec4c6fd9da208ee5a286caaf",
    "links.csv": "d5ccfdf196ebe5b7bbd09d505ea8ca263ecc33ae",
}
BASE = "https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/"
OFFICIAL_ZIP = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

def git_blob_sha(content: bytes):
    return hashlib.sha1(f"blob {len(content)}\0".encode() + content).hexdigest()

def download(url):
    with urlopen(Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=40) as r:
        return r.read()

def load_csv_files():
    local = [Path("ml-latest-small") / n for n in NAMES]
    if all(p.is_file() for p in local):
        payloads = {n: p.read_bytes() for n,p in zip(NAMES,local)}
        origin = "локальный каталог ml-latest-small"
    else:
        try:
            z = zipfile.ZipFile(io.BytesIO(download(OFFICIAL_ZIP)))
            payloads = {n: z.read("ml-latest-small/" + n) for n in NAMES}
            origin = "официальный архив GroupLens"
        except Exception:
            payloads = {n: download(BASE + n) for n in NAMES}
            origin = "публичное зеркало с SHA-контролем"
    for n,payload in payloads.items():
        actual = git_blob_sha(payload)
        assert actual == GIT_SHA[n], f"Контрольная сумма {n} изменилась: {actual}"
    return {n:pd.read_csv(io.BytesIO(blob)) for n,blob in payloads.items()}, origin

tables, origin = load_csv_files()
movies, ratings, tags, links = (tables[n] for n in NAMES)
print("Источник:", origin)
print("Контрольные суммы всех четырёх CSV совпали.")
display(pd.DataFrame({"table":NAMES,"rows":[len(tables[n]) for n in NAMES]}))

In [ ]:
# Данные не изменяются: проверки выполняются на raw.
ratings["datetime"] = pd.to_datetime(ratings["timestamp"],unit="s",utc=True)
tags["datetime"] = pd.to_datetime(tags["timestamp"],unit="s",utc=True)
movie_ids = set(movies["movieId"])
movie_genres = movies["genres"].fillna("(no genres listed)")
genre_count = movie_genres.str.split("|").map(lambda x: 0 if x == ["(no genres listed)"] else len(x))
title_year = pd.to_numeric(movies["title"].str.extract(r"\((\d{4})\)$")[0], errors="coerce")
rating_year = ratings["datetime"].dt.year
missing_year = int(title_year.isna().sum())
ratings_with_year = ratings.merge(
    pd.DataFrame({"movieId":movies["movieId"],"release_year":title_year}),
    on="movieId",how="left"
)
early_rating_count = int((ratings_with_year["datetime"].dt.year < ratings_with_year["release_year"]).sum())
collisions = int(movies["title"].duplicated().sum())
tie_counts = ratings.groupby(["userId","timestamp"]).size()
quality = {
    "movies_count": len(movies),
    "ratings_count": len(ratings),
    "tags_count": len(tags),
    "users_count": ratings["userId"].nunique(),
    "duplicate_movieId": int(movies["movieId"].duplicated().sum()),
    "duplicate_ratings_rows": int(ratings.drop(columns=["datetime"]).duplicated().sum()),
    "duplicate_user_movie": int(ratings.duplicated(["userId","movieId"]).sum()),
    "orphan_ratings": int((~ratings["movieId"].isin(movie_ids)).sum()),
    "orphan_tags": int((~tags["movieId"].isin(movie_ids)).sum()),
    "invalid_ratings": int((~ratings["rating"].between(.5,5) | ((ratings["rating"]*2)%1 != 0)).sum()),
    "missing_year_in_title": missing_year,
    "no_genres_listed": int(movie_genres.eq("(no genres listed)").sum()),
    "duplicate_titles": collisions,
    "missing_tmdbId": int(links["tmdbId"].isna().sum()),
    "empty_tags": int(tags["tag"].isna().sum() + tags["tag"].fillna("").astype(str).str.strip().eq("").sum() - tags["tag"].isna().sum()),
    "rating_before_release_year": early_rating_count,
    "tied_timestamp_groups": int((tie_counts>1).sum()),
    "ratings_in_tied_groups": int(tie_counts[tie_counts>1].sum()),
}
quality_df = pd.DataFrame({"check":list(quality),"observed":list(quality.values())})
display(quality_df)
quality_df.to_csv("lab05_quality_checks.csv",index=False)
print("Сохранено: lab05_quality_checks.csv")

In [ ]:
# Восемь сюжетных графиков на фактически загруженных строках.
genre_freq = movie_genres.str.split("|").explode()
genre_freq = genre_freq[genre_freq.ne("(no genres listed)")]
movie_pop = ratings["movieId"].value_counts()
split_groups = ratings.groupby(["userId","timestamp"],sort=True)
def per_user_split(df):
    result = np.full(len(df), "", dtype=object)
    # индексы исходных записей внутри пользователя сохраняем без перемешивания
    for uid, grp in df.groupby("userId"):
        grouped = list(grp.groupby("timestamp",sort=True))
        n = len(grouped)
        if n < 3: continue
        a = max(1,int(n*.60)); b=max(1,int(n*.20))
        if a+b >= n: a,b=n-2,1
        for i,(_,g) in enumerate(grouped):
            result[g.index.to_numpy()] = ("train" if i<a else "validation" if i<a+b else "test")
    return pd.Series(result,index=df.index)
split = per_user_split(ratings)
assert not split.eq("").any(), "Недостаточно уникальных временных групп хотя бы для одного пользователя"
gsets = {name:set(zip(ratings.loc[split==name,"userId"],ratings.loc[split==name,"timestamp"])) for name in ["train","validation","test"]}
assert not(gsets["train"]&gsets["validation"] or gsets["train"]&gsets["test"] or gsets["validation"]&gsets["test"])
print("Группы между train/validation/test не пересекаются")
issues = pd.Series({
    "Без жанров":quality["no_genres_listed"],
    "Без года":quality["missing_year_in_title"],
    "Нет TMDb ID":quality["missing_tmdbId"],
    "Повтор заголовка":quality["duplicate_titles"],
    "Рейтинг раньше релиза":quality["rating_before_release_year"]
})
charts = [
 ("01_genres",genre_freq.value_counts().head(10),"Фильмов с жанром"),
 ("02_ratings",ratings["rating"].value_counts().sort_index(),"Оценок"),
 ("03_release_decades",(title_year.dropna()//10*10).astype(int).value_counts().sort_index(),"Фильмов"),
 ("04_rating_years",rating_year.value_counts().sort_index(),"Оценок"),
 ("05_genres_per_movie",genre_count.value_counts().sort_index(),"Фильмов"),
 ("06_quality_issues",issues,"Объектов / аномалий"),
 ("07_split",split.value_counts().reindex(["train","validation","test"]),"Оценок"),
 ("08_popularity",pd.Series({
   "1 оценка":int((movie_pop==1).sum()),
   "2–4 оценки":int(movie_pop.between(2,4).sum()),
   "5+ оценок":int((movie_pop>=5).sum())
 }),"Фильмов"),
]
Path("figures").mkdir(exist_ok=True)
for name,series,unit in charts:
    fig,ax=plt.subplots(figsize=(10,4.5))
    series.plot(kind="bar",ax=ax,edgecolor="black")
    ax.set(title=name.replace("_"," ").upper(),ylabel=unit,xlabel="")
    plt.xticks(rotation=35,ha="right")
    fig.tight_layout()
    fig.savefig(f"figures/{name}.png",dpi=140)
    plt.show()
    plt.close(fig)
print("8 визуализаций сохранены в figures/*.png")

In [ ]:
# Примеры для ручной проверки, включая неудобные случаи.
interesting = [1,2,114335,122888,40697,140956,838,26958]
examples = movies[movies["movieId"].isin(interesting)].sort_values("movieId")
display(examples[["movieId","title","genres"]])
print("Проверяем не только типичные, но и безжанровые и дублирующиеся заголовки.")
print("Постеров в MovieLens нет; чужие изображения не импортируем без права.")

# Дополнительные проверки полей, способных служить proxy/источником утечки.
assert movies["movieId"].is_unique
assert ratings["movieId"].isin(movie_ids).all()
assert ratings["rating"].between(.5,5).all()
assert not ratings.duplicated(["userId","movieId"]).any()
print("PASS: ключи, ссылки, рейтинг и связность событий.")

report = pd.DataFrame({
    "data_version":["ml-latest-small-2018-09-26"],
    "source":["GroupLens / validated mirror"],
    "raw_movies":[len(movies)],
    "raw_ratings":[len(ratings)],
    "date_window_start":[ratings["datetime"].min().isoformat()],
    "date_window_end":[ratings["datetime"].max().isoformat()],
    "clean_policy":["retain raw; no silent deletion; annotate anomalies"],
    "train_rows":[int(split.eq("train").sum())],
    "validation_rows":[int(split.eq("validation").sum())],
    "test_rows":[int(split.eq("test").sum())],
})
display(report.T)
report.to_csv("lab05_data_card_metrics.csv",index=False)

## 5. Proxy-признаки и риски утечек

- `movieId`, `userId`, `timestamp` и имена файлов — идентификаторы или аудит, **не** числовые предикторы пользовательского вкуса.
- `rating` из validation/test — целевая переменная, не часть обучающего профиля.
- `title` содержит год, но строковые признаки должны быть отдельно обоснованы: год может проксировать эпоху или популярность.
- `tags.timestamp` следует сравнивать с моментом рекомендации: будущий тег недопустим.
- События `(userId,timestamp)` нужно целиком назначать одному split. Даже при непересекающихся группах остаются ограничения: неизвестна дата появления фильма в каталоге и не проверена переносимость на новых пользователей.

**Решение:** сохранить групповой chronological split из ЛР № 4, вычислять популярность только на train, TF-IDF IDF не подгонять по test и не интерпретировать корреляции как причинные эффекты.

## 6. Обновление Data Card и решение по проекту

**Обновлённый паспорт:** [data_card_update.md](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/data_card_update.md); подробная [таблица проверок и примеров](https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab05-eda-cinematch/quality_review.md).

**Data Card v1 → v1.1:** исходное описание источников и прав остаётся; дополнительно документируются проверенные агрегаты, 8 фигур с решениями, аномалии качества и контроль утечек. Цифры исходного каталога и агрегированного среза не приравниваются к результатам повторной локальной обработки. Выполняемые ячейки выше сохраняют `lab05_quality_checks.csv` и `lab05_data_card_metrics.csv`.

**Что делать дальше:** (1) построить корректный cold-start fallback для фильмов без жанров; (2) не заполнять отсутствующий год или tmdbId выдуманными значениями; (3) закрепить список непересекающихся временных групп; (4) отдельно изучать редкие жанры, малопопулярные фильмы, новинки и русскоязычные запросы.

**Критерии приёмки:** 8 осмысленных визуализаций с выводами, таблица проверок, неудобные примеры, explicit proxy review, контролируемое разделение и обновлённый Data Card. Для лабораторной не нужны обученные ML-модели или вымышленные CTR.